# Redis Stack

## Taller de Redis - Caso de uso: NexoRetail

> Autor: Marlon Cárdenas <br/>
> Año: 2026

NexoRetail es una plataforma omnicanal que vende tecnología, mobiliario y accesorios mediante web, aplicación móvil y tiendas físicas. La operación necesita responder en tiempo real a preguntas distintas:

* cuántos visitantes únicos han accedido hoy;
* qué productos generan más ingresos;
* qué clientes pertenecen a una campaña y a un segmento concreto;
* qué tienda está más cerca de un cliente;
* qué eventos siguen pendientes de procesamiento;
* cuál es la latencia p95 del servicio;
* qué productos y términos están ganando popularidad;
* cómo consultar el catálogo por texto, categoría, precio, stock y proveedor.

Redis resuelve cada necesidad mediante una estructura especializada. El criterio central del taller es este:

> El modelo se diseña a partir de las operaciones que debe ejecutar la aplicación, no a partir de una tabla genérica que intenta representar todo.

Al finalizar tendrás una capa de analítica operativa con datos sintéticos, coherentes y reproducibles.

## Objetivos

Al completar el cuaderno podrás:

1. explicar el modelo clave valor y la relación entre tipo y operación;
2. trabajar con Redis desde `redis-cli` y desde Python;
3. modelar perfiles, segmentos, rankings, actividad, eventos y series temporales;
4. diferenciar métricas exactas de estimaciones probabilísticas;
5. utilizar Redis JSON, Redis Search y Redis Time Series;
6. procesar eventos con Streams y grupos de consumidores;
7. reducir viajes de red mediante pipelines;
8. resolver operaciones críticas mediante Lua;
9. presentar resultados pequeños con pandas y Matplotlib;
10. reconocer cuándo Redis no es el almacenamiento adecuado.

## Mapa de estructuras

| Necesidad | Estructura | Propiedad útil |
|---|---|---|
| KPI, caché y configuración | `String`* | Acceso directo y expiración |
| Perfil de cliente | `Hash`* | Actualización por campo |
| Cola simple | `List`* | Inserción y extracción por extremos |
| Segmentos y campañas | `Set`* | Intersección, unión y diferencia |
| Ranking de ingresos | `Sorted Set`* | Orden exacto por puntuación |
| Usuarios activos | `Bitmap` | Un bit por usuario y día |
| Puntuación compacta | `Bitfield` | Enteros pequeños empaquetados |
| Visitantes únicos | `HyperLogLog` | Cardinalidad aproximada |
| Tiendas cercanas | `Geospatial` | Radio y distancia |
| Registro de actividad | `Stream`* | Historial, grupos y confirmación |
| Catálogo anidado | `JSON`* | Objetos y arrays |
| Consulta del catálogo | `Redis Search`* | Texto, filtros y agregaciones |
| Ingresos por intervalo | `Time Series` | Retención y ventanas temporales |
| Duplicados | `Bloom Filter` | Pertenencia aproximada |
| Bloqueos eliminables | `Cuckoo Filter` | Pertenencia aproximada con borrado |
| Frecuencia de vistas | `Count-Min Sketch` | Conteo aproximado a gran escala |
| Tendencias | `Top-K` | Elementos principales |
| Percentiles | `t-digest` | Cuantiles aproximados |
| Similitud | `Vector Set`*, `Redis 8` | Vecinos por distancia vectorial |

## Índice

1. Preparación del entorno
2. Modelo mental y convenciones
3. Tipos fundamentales desde la terminal
4. Redis Stack desde la terminal
5. Vector Sets en Redis 8
6. Preparación de Python
7. Construcción del conjunto de datos
8. Carga de dimensiones
9. Ingestión de eventos y métricas
10. Análisis con Redis y pandas
11. Pipelines, atomicidad y Lua
12. Procesamiento con Streams
13. Caso integrado
14. Práctica final
15. Criterios de modelado

# 1. Preparación del entorno

El taller principal utiliza `redis/redis-stack:latest`, que incluye Redis Stack y Redis Insight. Esta imagen corresponde a la línea Redis Stack 7.x e incorpora Redis Search, JSON, Time Series y estructuras probabilísticas.

Redis 8 integra estas capacidades en Redis Open Source. La sección de Vector Sets utiliza un segundo contenedor Redis 8 porque ese tipo no forma parte de Redis Stack 7.x.

### Requisitos

* Docker Desktop o Docker Engine.
* Python 3.10 o superior.
* Jupyter Notebook o JupyterLab.
* Puertos `6379`, `6380` y `8001` libres.

No conectes este cuaderno a una instancia compartida. El espacio de trabajo utiliza el prefijo `lab:` y elimina esas claves al comenzar.

## 1.1. Iniciar Redis Stack

Ejecuta en PowerShell, Terminal o bash:

```bash
docker run -d --name redis-stack-lab -p 6379:6379 -p 8001:8001 -v redis-stack-lab-data:/data -e REDIS_ARGS="--appendonly yes" redis/redis-stack:latest
```

El comando:

1. crea el contenedor `redis-stack-lab`;
2. publica Redis en `localhost:6379`;
3. publica Redis Insight en `http://localhost:8001`;
4. activa AOF;
5. guarda los datos en un volumen Docker.

Comprueba el contenedor:

```bash
docker ps --filter name=redis-stack-lab
```

Si ya existe pero está detenido:

```bash
docker start redis-stack-lab
```

Si necesitas recrearlo:

```bash
docker rm -f redis-stack-lab
```

## 1.2. Conectar con `redis-cli`

```bash
docker exec -it redis-stack-lab redis-cli
```

Comprueba la conexión y las capacidades:

```redis
PING
MODULE LIST
COMMAND INFO JSON.SET FT.CREATE TS.ADD BF.ADD
INFO server
```

`PING` debe devolver `PONG`. `MODULE LIST` debe mostrar los componentes de búsqueda, JSON, series temporales y probabilísticos.

Abre Redis Insight en `http://localhost:8001` y conecta con `localhost:6379`. Utiliza el navegador de claves para observar el tipo, contenido, TTL y memoria ocupada.

# 2. Modelo mental y convenciones

Redis almacena claves asociadas a valores tipados:

```text
clave -> valor de un tipo concreto
```

Ejemplos del taller:

```text
lab:customer:C1001                 -> hash
lab:segment:premium                -> set
lab:ranking:products:revenue       -> zset
lab:events:retail                  -> stream
lab:product:P001                   -> json
lab:ts:revenue:web                 -> timeseries
```

Convención recomendada:

```text
entorno:dominio:entidad:identificador:atributo
```

Inspección:

```redis
DBSIZE
SCAN 0 MATCH lab:* COUNT 100
TYPE lab:customer:C1001
MEMORY USAGE lab:customer:C1001
TTL lab:session:C1001
```

Evita `KEYS *` en sistemas reales. `KEYS` recorre el espacio completo de claves de forma bloqueante. `SCAN` avanza de forma incremental.

En el contenedor aislado puedes reiniciar manualmente con:

```redis
FLUSHDB
```

No uses `FLUSHDB` en una instancia compartida.

# 3. Tipos fundamentales desde la terminal

Ejecuta los comandos siguientes dentro de `redis-cli`. Todos los ejemplos pertenecen al mismo sistema y resuelven necesidades operativas concretas.

## 3.1. Strings: KPI, caché, configuración y límites

Un String almacena texto, números o bytes. Redis interpreta el contenido como número cuando ejecutas operaciones numéricas.

### Ingresos del día

```redis
SET lab:kpi:revenue:2026-06-14 18320.45
INCRBYFLOAT lab:kpi:revenue:2026-06-14 249.90
GET lab:kpi:revenue:2026-06-14
```

### Resultado de panel con caducidad

```redis
SET lab:cache:dashboard:executive '{"revenue":18570.35,"orders":74}' EX 300
TTL lab:cache:dashboard:executive
GET lab:cache:dashboard:executive
```

### Funcionalidad temporal

```redis
SET lab:feature:recommendations enabled EX 3600
GET lab:feature:recommendations
```

### Contador de solicitudes

```redis
SET lab:rate:user:C1001 1 EX 60 NX
INCR lab:rate:user:C1001
TTL lab:rate:user:C1001
```

Este último patrón ilustra una ventana temporal, pero no garantiza que incremento y expiración formen una operación indivisible. La sección de Lua implementa la variante atómica.

Usa String cuando la lectura principal obtiene o reemplaza un valor completo.

## 3.2. Hashes: perfil operativo de cliente

Un Hash representa un registro plano y permite modificar campos independientes.

```redis
HSET lab:customer:C1001 name "Ana Torres" city "Madrid" segment "premium" purchases 4 total_spent 825.50 churn_score 0.18
HGETALL lab:customer:C1001
HMGET lab:customer:C1001 name segment total_spent
HINCRBY lab:customer:C1001 purchases 1
HINCRBYFLOAT lab:customer:C1001 total_spent 129.90
HEXISTS lab:customer:C1001 churn_score
HLEN lab:customer:C1001
```

El Hash funciona como una vista de cliente para lecturas por identificador. No representa el historial completo de compras.

## 3.3. Lists: cola operativa simple

NexoRetail recibe órdenes que deben reconciliarse con un proveedor externo:

```redis
RPUSH lab:queue:reconciliation order:O9001 order:O9002 order:O9003
LLEN lab:queue:reconciliation
LRANGE lab:queue:reconciliation 0 -1
LPOP lab:queue:reconciliation
BLPOP lab:queue:reconciliation 5
```

Una pila de acciones recientes utiliza el extremo contrario:

```redis
LPUSH lab:actions:C1001 login view:P001 add_to_cart:P001
LPOP lab:actions:C1001
```

Una List no ofrece confirmación ni grupos de consumidores. Si el trabajador retira una orden y falla, el elemento ya no está disponible. Para eventos críticos usa Streams.

## 3.4. Sets: campañas y segmentos

```redis
SADD lab:campaign:summer C1001 C1002 C1003 C1004
SADD lab:segment:premium C1001 C1003 C1005 C1008
SINTER lab:campaign:summer lab:segment:premium
SUNION lab:campaign:summer lab:segment:premium
SDIFF lab:campaign:summer lab:segment:premium
SISMEMBER lab:campaign:summer C1002
SCARD lab:campaign:summer
```

Las operaciones se ejecutan dentro de Redis y devuelven únicamente el resultado. Los miembros son únicos y el resultado es exacto.

## 3.5. Sorted Sets: ranking de productos

```redis
ZADD lab:ranking:products:revenue 2450.50 P001 970.20 P002 3820.00 P003
ZINCRBY lab:ranking:products:revenue 250.00 P002
ZREVRANGE lab:ranking:products:revenue 0 -1 WITHSCORES
ZREVRANGE lab:ranking:products:revenue 0 2 WITHSCORES
ZREVRANK lab:ranking:products:revenue P002
ZRANGEBYSCORE lab:ranking:products:revenue 1000 3000 WITHSCORES
```

La puntuación puede representar ingresos, margen, prioridad, riesgo o tiempo. Un Set responde pertenencia. Un Sorted Set responde posición y rango.

## 3.6. Bitmaps: actividad y retención

Las posiciones `1`, `2`, `3` y `4` representan identificadores internos compactos de clientes.

```redis
SETBIT lab:active:2026-06-13 1 1
SETBIT lab:active:2026-06-13 2 1
SETBIT lab:active:2026-06-13 4 1
SETBIT lab:active:2026-06-14 2 1
SETBIT lab:active:2026-06-14 3 1
SETBIT lab:active:2026-06-14 4 1
BITCOUNT lab:active:2026-06-13
BITCOUNT lab:active:2026-06-14
GETBIT lab:active:2026-06-14 3
BITOP AND lab:active:retained lab:active:2026-06-13 lab:active:2026-06-14
BITCOUNT lab:active:retained
BITOP OR lab:active:either lab:active:2026-06-13 lab:active:2026-06-14
BITCOUNT lab:active:either
```

La posición máxima determina el tamaño. No uses directamente identificadores dispersos de miles de millones.

## 3.7. Bitfield: puntuación compacta de riesgo

Cada cliente recibe una puntuación entre 0 y 255:

```redis
BITFIELD lab:risk:scores SET u8 #1 72 SET u8 #2 34
BITFIELD lab:risk:scores GET u8 #1 GET u8 #2
BITFIELD lab:risk:scores OVERFLOW SAT INCRBY u8 #1 20
BITFIELD lab:risk:scores OVERFLOW SAT INCRBY u8 #1 250
```

Bitfield reduce memoria cuando existen millones de enteros pequeños y el esquema binario permanece estable. Un Hash ofrece mejor legibilidad cuando la escala no exige esta optimización.

## 3.8. HyperLogLog: visitantes únicos

```redis
PFADD lab:visitors:2026-06-13 visitor:1 visitor:2 visitor:3 visitor:4
PFADD lab:visitors:2026-06-14 visitor:2 visitor:3 visitor:5 visitor:6
PFCOUNT lab:visitors:2026-06-13
PFCOUNT lab:visitors:2026-06-14
PFMERGE lab:visitors:period lab:visitors:2026-06-13 lab:visitors:2026-06-14
PFCOUNT lab:visitors:period
```

HyperLogLog estima cardinalidad y no permite recuperar miembros. Usa Set cuando necesitas exactitud o los identificadores.

## 3.9. Geospatial: tienda más cercana

```redis
GEOADD lab:stores -3.7038 40.4168 store:MAD-CENTRO -3.6883 40.4650 store:MAD-NORTE 2.1630 41.3890 store:BCN-EIXAMPLE -0.3763 39.4699 store:VLC-CENTRO
GEOSEARCH lab:stores FROMLONLAT -3.7038 40.4168 BYRADIUS 10 km WITHDIST ASC
GEODIST lab:stores store:MAD-CENTRO store:MAD-NORTE km
```

La distancia geográfica sirve para preseleccionar tiendas. No equivale al tiempo real de conducción.

## 3.10. Streams: eventos y consumidores

```redis
XADD lab:events:retail * event_id EVT-1 visitor_id visitor:1 event view product_id P001 amount 0 channel web
XADD lab:events:retail * event_id EVT-2 visitor_id visitor:1 event purchase product_id P001 amount 1199 channel web
XRANGE lab:events:retail - +
XREVRANGE lab:events:retail + - COUNT 2
XLEN lab:events:retail
XTRIM lab:events:retail MAXLEN ~ 10000
XGROUP CREATE lab:events:retail analytics 0 MKSTREAM
XREADGROUP GROUP analytics analyst-1 COUNT 10 STREAMS lab:events:retail >
XACK lab:events:retail analytics <identificador>
XINFO GROUPS lab:events:retail
XPENDING lab:events:retail analytics
```

| Propiedad | List | Stream |
|---|---:|---:|
| Orden | Sí | Sí |
| Historial después de leer | No necesariamente | Sí |
| Identificador por entrada | No | Sí |
| Grupos de consumidores | No | Sí |
| Confirmación | No | Sí |
| Reprocesamiento | Manual | Mediante pendientes |

# 4. Redis Stack desde la terminal

## 4.1. JSON: catálogo estructurado

```redis
JSON.SET lab:product:P001 $ '{"name":"Portátil Atlas 14","category":"informatica","price":1199.0,"stock":18,"tags":["movilidad","premium"],"supplier":{"id":"SUP-07","country":"ES"},"description":"Portátil ligero para trabajo profesional y movilidad"}'
JSON.SET lab:product:P002 $ '{"name":"Monitor Nexo 27","category":"informatica","price":349.0,"stock":42,"tags":["oficina","monitor"],"supplier":{"id":"SUP-03","country":"DE"},"description":"Monitor QHD de 27 pulgadas para productividad"}'
JSON.SET lab:product:P003 $ '{"name":"Silla Ergo Pro","category":"oficina","price":425.0,"stock":12,"tags":["ergonomia","oficina"],"supplier":{"id":"SUP-09","country":"ES"},"description":"Silla ergonómica con soporte lumbar regulable"}'
JSON.GET lab:product:P001 $
JSON.GET lab:product:P001 $.supplier.country
JSON.NUMINCRBY lab:product:P001 $.stock -1
JSON.ARRAPPEND lab:product:P001 $.tags '"oferta"'
```

Usa Hash para registros planos. Usa JSON cuando necesitas arrays, objetos anidados y búsqueda por campos.

## 4.2. Redis Search: catálogo consultable

```redis
FT.CREATE lab:idx:products ON JSON PREFIX 1 lab:product: LANGUAGE spanish SCHEMA $.name AS name TEXT WEIGHT 3.0 $.description AS description TEXT $.category AS category TAG $.price AS price NUMERIC SORTABLE $.stock AS stock NUMERIC SORTABLE $.supplier.country AS supplier_country TAG
FT.SEARCH lab:idx:products "ergonómica" RETURN 4 name category price stock
FT.SEARCH lab:idx:products "@category:{informatica} @price:[300 1500] @stock:[1 +inf]" RETURN 4 name category price stock SORTBY price ASC DIALECT 2
FT.SEARCH lab:idx:products "@supplier_country:{ES}" RETURN 3 name category price DIALECT 2
FT.AGGREGATE lab:idx:products "*" GROUPBY 1 @category REDUCE COUNT 0 AS products REDUCE AVG 1 @price AS average_price SORTBY 2 @average_price DESC
```

Redis Search permite localizar documentos sin conocer sus claves. Para históricos extensos y joins complejos sigue siendo necesario un sistema analítico.

## 4.3. Time Series: ingresos por intervalo

```redis
TS.CREATE lab:ts:revenue:web RETENTION 1209600000 DUPLICATE_POLICY SUM LABELS metric revenue channel web
TS.ADD lab:ts:revenue:web 1780272000000 1199.00
TS.ADD lab:ts:revenue:web 1780272060000 349.00
TS.ADD lab:ts:revenue:web 1780272120000 129.90
TS.RANGE lab:ts:revenue:web - +
TS.RANGE lab:ts:revenue:web - + AGGREGATION sum 300000
TS.GET lab:ts:revenue:web
TS.CREATE lab:ts:revenue:web:hour RETENTION 7776000000 LABELS metric revenue_hour channel web
TS.CREATERULE lab:ts:revenue:web lab:ts:revenue:web:hour AGGREGATION sum 3600000
```

Time Series evita crear una clave por observación y ofrece retención, etiquetas, agregaciones y reglas de compactación.

## 4.4. Estructuras probabilísticas

### Bloom Filter

```redis
BF.RESERVE lab:dedupe:events 0.001 100000
BF.ADD lab:dedupe:events EVT-1001
BF.ADD lab:dedupe:events EVT-1002
BF.EXISTS lab:dedupe:events EVT-1001
BF.EXISTS lab:dedupe:events EVT-9999
```

Un `0` implica ausencia. Un `1` implica presencia probable.

### Cuckoo Filter

```redis
CF.RESERVE lab:blocked:transactions 10000
CF.ADD lab:blocked:transactions TX-44
CF.EXISTS lab:blocked:transactions TX-44
CF.DEL lab:blocked:transactions TX-44
```

### Count-Min Sketch

```redis
CMS.INITBYPROB lab:frequency:product_views 0.001 0.01
CMS.INCRBY lab:frequency:product_views P001 120 P002 80 P003 25
CMS.QUERY lab:frequency:product_views P001 P002 P003
```

### Top-K

```redis
TOPK.RESERVE lab:trending:products 5 2000 7 0.925
TOPK.INCRBY lab:trending:products P001 120 P002 80 P003 25 P004 90 P005 60 P006 45
TOPK.LIST lab:trending:products WITHCOUNT
```

### t-digest

```redis
TDIGEST.CREATE lab:latency:api COMPRESSION 100
TDIGEST.ADD lab:latency:api 90 110 105 125 160 200 95 500 130 115
TDIGEST.QUANTILE lab:latency:api 0.50 0.95 0.99
TDIGEST.MIN lab:latency:api
TDIGEST.MAX lab:latency:api
```

Estas estructuras intercambian exactitud por memoria y velocidad. No las uses cuando un falso positivo o una estimación incorrecta produzca una decisión crítica.

# 5. Vector Sets en Redis 8

Inicia un segundo contenedor:

```bash
docker run -d --name redis8-vector-lab -p 6380:6379 -v redis8-vector-lab-data:/data redis:8 redis-server --appendonly yes
```

Conecta:

```bash
docker exec -it redis8-vector-lab redis-cli
```

Ejemplo manual:

```redis
VADD lab:product:vectors VALUES 4 0.95 0.80 0.15 0.10 P001
VADD lab:product:vectors VALUES 4 0.85 0.75 0.20 0.15 P002
VADD lab:product:vectors VALUES 4 0.10 0.25 0.90 0.75 P003
VADD lab:product:vectors VALUES 4 0.20 0.35 0.80 0.65 P004
VSIM lab:product:vectors VALUES 4 0.90 0.78 0.18 0.12 COUNT 3 WITHSCORES
VSETATTR lab:product:vectors P001 '{"category":"informatica","price":1199}'
VSETATTR lab:product:vectors P002 '{"category":"informatica","price":349}'
VSIM lab:product:vectors VALUES 4 0.90 0.78 0.18 0.12 COUNT 3 FILTER '.price < 1000' WITHSCORES
```

En producción, los vectores manuales se sustituyen por embeddings generados a partir de texto, comportamiento o atributos.

# 6. Preparación de Python

Ejecuta las celdas en orden. La conexión principal apunta a Redis Stack en `6379`.

In [ ]:
%pip install -q -U redis pandas numpy matplotlib

In [ ]:
from __future__ import annotations

import json
import math
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import redis
from IPython.display import display
from redis import Redis
from redis.exceptions import ConnectionError, ResponseError

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS8_PORT = 6380
PREFIX = "lab:"
INDEX_NAME = "lab:idx:products"

r = Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    decode_responses=True,
    socket_connect_timeout=5,
    socket_timeout=10,
)

print("PING:", r.ping())
print("redis-py:", redis.__version__)

## 6.1. Verificación de capacidades

El cuaderno detiene la ejecución si la instancia no contiene los comandos necesarios.

In [ ]:
required_commands = [
    "JSON.SET",
    "FT.CREATE",
    "TS.ADD",
    "BF.ADD",
    "CMS.INCRBY",
    "TOPK.INCRBY",
    "TDIGEST.ADD",
]

missing = []
for command_name in required_commands:
    info = r.execute_command("COMMAND", "INFO", command_name)
    if not info or info[0] is None:
        missing.append(command_name)

if missing:
    raise RuntimeError("Faltan comandos: " + ", ".join(missing))

print("Capacidades verificadas:", ", ".join(required_commands))

## 6.2. Limpieza segura

Se eliminan únicamente claves `lab:*`. El índice de búsqueda se elimina por separado.

In [ ]:
def delete_by_pattern(client: Redis, pattern: str, batch_size: int = 500) -> int:
    deleted = 0
    cursor = 0
    while True:
        cursor, keys = client.scan(cursor=cursor, match=pattern, count=batch_size)
        if keys:
            deleted += client.delete(*keys)
        if cursor == 0:
            break
    return deleted


def drop_index_if_exists(client: Redis, index_name: str) -> None:
    try:
        client.execute_command("FT.DROPINDEX", index_name)
    except ResponseError as exc:
        message = str(exc).lower()
        if "unknown index" not in message and "no such index" not in message:
            raise


drop_index_if_exists(r, INDEX_NAME)
print("Claves eliminadas:", delete_by_pattern(r, "lab:*"))

# 7. Construcción del conjunto de datos

Los datos son sintéticos, pero mantienen relaciones y distribuciones coherentes:

* productos con categoría, precio, stock, proveedor, etiquetas y descripción;
* clientes con ciudad, segmento, gasto y riesgo de abandono;
* visitantes registrados y anónimos;
* compras con cantidad, descuento e importe;
* latencia dependiente del canal;
* demanda desigual entre productos;
* catorce días de actividad.

In [ ]:
products = pd.DataFrame([
    {"product_id":"P001","name":"Portátil Atlas 14","category":"informatica","price":1199.0,"stock":18,"supplier_id":"SUP-07","supplier_country":"ES","tags":["movilidad","premium"],"description":"Portátil ligero de 14 pulgadas para trabajo profesional, análisis de datos y movilidad."},
    {"product_id":"P002","name":"Monitor Nexo 27","category":"informatica","price":349.0,"stock":42,"supplier_id":"SUP-03","supplier_country":"DE","tags":["oficina","monitor"],"description":"Monitor QHD de 27 pulgadas con base regulable para productividad y programación."},
    {"product_id":"P003","name":"Silla Ergo Pro","category":"oficina","price":425.0,"stock":12,"supplier_id":"SUP-09","supplier_country":"ES","tags":["ergonomia","oficina"],"description":"Silla ergonómica con soporte lumbar, reposabrazos y ajuste de profundidad."},
    {"product_id":"P004","name":"Mesa Elevable Motion","category":"oficina","price":599.0,"stock":9,"supplier_id":"SUP-09","supplier_country":"ES","tags":["ergonomia","mesa"],"description":"Mesa eléctrica regulable en altura para alternar trabajo sentado y de pie."},
    {"product_id":"P005","name":"Auriculares Quiet One","category":"audio","price":279.0,"stock":31,"supplier_id":"SUP-11","supplier_country":"JP","tags":["audio","movilidad"],"description":"Auriculares inalámbricos con cancelación de ruido para oficina y viajes."},
    {"product_id":"P006","name":"Teclado Mecánico Code","category":"accesorios","price":139.0,"stock":55,"supplier_id":"SUP-04","supplier_country":"NL","tags":["teclado","programacion"],"description":"Teclado mecánico compacto con distribución española y conexión dual."},
    {"product_id":"P007","name":"Ratón Precision MX","category":"accesorios","price":89.9,"stock":68,"supplier_id":"SUP-04","supplier_country":"NL","tags":["raton","productividad"],"description":"Ratón inalámbrico ergonómico con desplazamiento de precisión."},
    {"product_id":"P008","name":"Dock USB-C Connect","category":"accesorios","price":159.0,"stock":37,"supplier_id":"SUP-02","supplier_country":"CN","tags":["conectividad","movilidad"],"description":"Base USB-C con vídeo, red, carga y puertos para un puesto híbrido."},
    {"product_id":"P009","name":"Webcam Studio 4K","category":"video","price":199.0,"stock":24,"supplier_id":"SUP-05","supplier_country":"TW","tags":["video","teletrabajo"],"description":"Cámara 4K con encuadre automático y micrófonos para reuniones profesionales."},
    {"product_id":"P010","name":"Micrófono Voice Pro","category":"audio","price":179.0,"stock":21,"supplier_id":"SUP-05","supplier_country":"TW","tags":["audio","streaming"],"description":"Micrófono USB para formación, reuniones, grabación y retransmisión."},
    {"product_id":"P011","name":"Tablet Nova 11","category":"movilidad","price":649.0,"stock":16,"supplier_id":"SUP-12","supplier_country":"KR","tags":["tablet","movilidad"],"description":"Tablet de 11 pulgadas con lápiz digital para movilidad y toma de notas."},
    {"product_id":"P012","name":"Mochila Urban Tech","category":"movilidad","price":79.0,"stock":73,"supplier_id":"SUP-08","supplier_country":"PT","tags":["mochila","portatil"],"description":"Mochila impermeable con compartimento acolchado para portátil y accesorios."},
    {"product_id":"P013","name":"Lámpara Focus","category":"hogar","price":69.0,"stock":46,"supplier_id":"SUP-10","supplier_country":"ES","tags":["iluminacion","oficina"],"description":"Lámpara de escritorio regulable con temperatura de color adaptable."},
    {"product_id":"P014","name":"Servidor MiniLab","category":"informatica","price":899.0,"stock":7,"supplier_id":"SUP-06","supplier_country":"DE","tags":["servidor","laboratorio"],"description":"Equipo compacto para laboratorios, virtualización y servicios domésticos."},
    {"product_id":"P015","name":"Disco SSD Vault 2TB","category":"almacenamiento","price":169.0,"stock":64,"supplier_id":"SUP-01","supplier_country":"US","tags":["ssd","almacenamiento"],"description":"Unidad SSD externa de dos terabytes con cifrado y conexión USB-C."},
])
products["price"] = products["price"].astype(float)
products["stock"] = products["stock"].astype(int)
display(products[["product_id","name","category","price","stock"]])

In [ ]:
rng = np.random.default_rng(20260622)
first_names = ["Ana","Carlos","Laura","David","Marta","Javier","Lucía","Pablo","Elena","Sergio","Nuria","Álvaro","Irene","Miguel","Clara","Raúl","Patricia","Diego","Beatriz","Andrés","Sofía","Mario","Carmen","Héctor","Eva","Rubén","Alicia","Marcos","Teresa","Óscar"]
last_names = ["Torres","Ruiz","Pérez","Martín","García","Sanz","Moreno","Vega","Navarro","Castro","Ramos","Gil","Romero","Molina","Ortega","Santos","Iglesias","Cano","Prieto","Rey","Méndez","Vidal","Suárez","León","Calvo","Lozano","Pastor","Serrano","Blanco","Nieto"]
cities = ["Madrid","Barcelona","Valencia","Sevilla","Bilbao"]
segments = ["premium","standard","new","at_risk"]

rows = []
for i in range(30):
    segment = rng.choice(segments, p=[0.22,0.48,0.18,0.12])
    base = {"premium":0.12,"standard":0.28,"new":0.20,"at_risk":0.72}[segment]
    rows.append({
        "customer_id": f"C{1001+i}",
        "bitmap_id": i+1,
        "name": f"{first_names[i]} {last_names[i]}",
        "city": rng.choice(cities, p=[0.42,0.20,0.16,0.13,0.09]),
        "segment": segment,
        "age": int(rng.integers(23,67)),
        "signup_date": (pd.Timestamp("2026-06-01")-pd.to_timedelta(int(rng.integers(30,1500)),unit="D")).date().isoformat(),
        "purchases": int(rng.poisson(8 if segment=="premium" else 4)),
        "total_spent": round(max(0.0,rng.gamma(3.0 if segment=="premium" else 1.8,260.0)),2),
        "churn_score": round(float(np.clip(rng.normal(base,0.08),0.01,0.99)),3),
    })
customers = pd.DataFrame(rows)
display(customers.head(10))

In [ ]:
stores = pd.DataFrame([
    {"store_id":"MAD-CENTRO","name":"Madrid Centro","city":"Madrid","longitude":-3.7038,"latitude":40.4168},
    {"store_id":"MAD-NORTE","name":"Madrid Norte","city":"Madrid","longitude":-3.6883,"latitude":40.4650},
    {"store_id":"BCN-EIXAMPLE","name":"Barcelona Eixample","city":"Barcelona","longitude":2.1630,"latitude":41.3890},
    {"store_id":"VLC-CENTRO","name":"Valencia Centro","city":"Valencia","longitude":-0.3763,"latitude":39.4699},
    {"store_id":"SVQ-NERVION","name":"Sevilla Nervión","city":"Sevilla","longitude":-5.9771,"latitude":37.3828},
    {"store_id":"BIO-ABANDO","name":"Bilbao Abando","city":"Bilbao","longitude":-2.9253,"latitude":43.2630},
])
display(stores)

## 7.1. Generación de eventos

El flujo contiene `view`, `add_to_cart`, `purchase`, `search` y `login`. Los visitantes anónimos participan en navegación y búsqueda. Las compras se asignan a clientes registrados. El importe depende del precio, la cantidad y un descuento ocasional.

In [ ]:
N_EVENTS = 2500
START = pd.Timestamp("2026-06-01 00:00:00", tz="Europe/Madrid")
END = pd.Timestamp("2026-06-15 00:00:00", tz="Europe/Madrid")
registered = customers["customer_id"].tolist()
anonymous = [f"A{i:04d}" for i in range(1,181)]
visitors = registered + anonymous
product_ids = products["product_id"].tolist()
product_price = products.set_index("product_id")["price"].to_dict()
popularity = np.array([0.13,0.11,0.07,0.05,0.10,0.08,0.09,0.07,0.05,0.04,0.05,0.06,0.03,0.03,0.04])
popularity = popularity / popularity.sum()
search_terms = ["portátil para trabajo","monitor oficina","silla ergonómica","mesa elevable","auriculares cancelación ruido","teclado mecánico","ratón inalámbrico","dock usb c","webcam reuniones","ssd externo","servidor laboratorio","mochila portátil"]

seconds = int((END-START).total_seconds())
timestamps = START + pd.to_timedelta(rng.integers(0,seconds,size=N_EVENTS),unit="s")
event_types = rng.choice(["view","add_to_cart","purchase","search","login"],size=N_EVENTS,p=[0.50,0.14,0.11,0.18,0.07])
channels = rng.choice(["web","app","store"],size=N_EVENTS,p=[0.52,0.38,0.10])

rows = []
for i in range(N_EVENTS):
    event_type = event_types[i]
    channel = channels[i]
    visitor_id = rng.choice(registered if event_type=="purchase" else visitors)
    customer_id = visitor_id if visitor_id.startswith("C") else ""
    product_id = rng.choice(product_ids,p=popularity) if event_type in {"view","add_to_cart","purchase"} else ""
    quantity = int(rng.integers(1,4)) if event_type=="purchase" else 0
    discount = float(rng.choice([0.0,0.05,0.10,0.15],p=[0.62,0.18,0.15,0.05]))
    amount = round(product_price[product_id]*quantity*(1-discount),2) if event_type=="purchase" else 0.0
    base_latency = {"web":145,"app":115,"store":185}[channel]
    latency_ms = int(max(20,rng.lognormal(mean=math.log(base_latency),sigma=0.38)))
    rows.append({
        "event_id":f"EVT-{i+1:06d}","timestamp":timestamps[i],"visitor_id":visitor_id,"customer_id":customer_id,
        "event_type":event_type,"product_id":product_id,"quantity":quantity,"amount":amount,"discount":discount,
        "channel":channel,"latency_ms":latency_ms,"search_term":rng.choice(search_terms) if event_type=="search" else "",
        "store_id":rng.choice(stores["store_id"]) if channel=="store" else "",
    })

events = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
events["timestamp_ms"] = events["timestamp"].astype("int64") // 1_000_000
events["event_date"] = events["timestamp"].dt.strftime("%Y-%m-%d")
events["minute"] = events["timestamp"].dt.floor("min")

display(events.head(12))
print("Periodo:", events["timestamp"].min(), "a", events["timestamp"].max())

In [ ]:
assert events["event_id"].is_unique
assert events.loc[events["event_type"]=="purchase","amount"].gt(0).all()
assert events.loc[events["event_type"]!="purchase","amount"].eq(0).all()
assert set(events["product_id"].unique()) - {""} <= set(products["product_id"])

summary = events.groupby(["event_type","channel"],as_index=False).agg(
    events=("event_id","count"),
    revenue=("amount","sum"),
    p95_latency=("latency_ms",lambda x:x.quantile(0.95)),
)
display(summary)

ax = events["event_type"].value_counts().plot(kind="bar",figsize=(9,4),title="Distribución de eventos")
ax.set_xlabel("Tipo")
ax.set_ylabel("Eventos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 8. Carga de dimensiones

Se utilizarán varias representaciones:

* clientes como Hashes;
* segmentos como Sets;
* valor acumulado como Sorted Set;
* productos como JSON;
* stock como Hash operativo;
* tiendas como estructura geoespacial.

La duplicación entre JSON e inventario es intencional. El catálogo sirve a consultas ricas. El Hash de inventario sirve a actualizaciones atómicas.

In [ ]:
pipe = r.pipeline(transaction=False)
for row in customers.itertuples(index=False):
    pipe.hset(f"lab:customer:{row.customer_id}",mapping={
        "customer_id":row.customer_id,"bitmap_id":int(row.bitmap_id),"name":row.name,"city":row.city,
        "segment":row.segment,"age":int(row.age),"signup_date":row.signup_date,"purchases":int(row.purchases),
        "total_spent":float(row.total_spent),"churn_score":float(row.churn_score),
    })
    pipe.sadd(f"lab:segment:{row.segment}",row.customer_id)
    pipe.zadd("lab:ranking:customers:total_spent",{row.customer_id:float(row.total_spent)})
pipe.execute()

print(r.hgetall("lab:customer:C1001"))
print("Premium:",r.scard("lab:segment:premium"))

In [ ]:
pipe = r.pipeline(transaction=False)
for row in products.itertuples(index=False):
    document = {
        "product_id":row.product_id,"name":row.name,"category":row.category,"price":float(row.price),"stock":int(row.stock),
        "tags":list(row.tags),"supplier":{"id":row.supplier_id,"country":row.supplier_country},"description":row.description,
    }
    pipe.execute_command("JSON.SET",f"lab:product:{row.product_id}","$",json.dumps(document,ensure_ascii=False))
    pipe.hset("lab:inventory:stock",row.product_id,int(row.stock))
    pipe.hset("lab:inventory:price",row.product_id,float(row.price))
pipe.execute()

print(json.dumps(json.loads(r.execute_command("JSON.GET","lab:product:P001","$")),ensure_ascii=False,indent=2))

In [ ]:
r.execute_command(
    "FT.CREATE",INDEX_NAME,"ON","JSON","PREFIX",1,"lab:product:","LANGUAGE","spanish","SCHEMA",
    "$.product_id","AS","product_id","TAG",
    "$.name","AS","name","TEXT","WEIGHT",3.0,
    "$.description","AS","description","TEXT",
    "$.category","AS","category","TAG",
    "$.price","AS","price","NUMERIC","SORTABLE",
    "$.stock","AS","stock","NUMERIC","SORTABLE",
    "$.supplier.country","AS","supplier_country","TAG",
)
print("Indexados:",r.execute_command("FT.SEARCH",INDEX_NAME,"*","LIMIT",0,0,"DIALECT",2)[0])

In [ ]:
pipe = r.pipeline(transaction=False)
geo_args = ["GEOADD","lab:stores"]
for row in stores.itertuples(index=False):
    geo_args.extend([float(row.longitude),float(row.latitude),f"store:{row.store_id}"])
    pipe.hset(f"lab:store:{row.store_id}",mapping={
        "store_id":row.store_id,"name":row.name,"city":row.city,
        "longitude":float(row.longitude),"latitude":float(row.latitude),
    })
pipe.execute_command(*geo_args)
pipe.execute()

assert r.type("lab:customer:C1001")=="hash"
assert r.hlen("lab:inventory:stock")==len(products)
assert r.zcard("lab:stores")==len(stores)
print("Dimensiones validadas")

# 9. Ingestión de eventos y métricas

El Stream conserva cada evento. Las estructuras derivadas responden consultas concretas sin recorrer el historial completo.

In [ ]:
STREAM_KEY = "lab:events:retail"
stream_ids = []

for start in range(0,len(events),250):
    pipe = r.pipeline(transaction=False)
    for row in events.iloc[start:start+250].itertuples(index=False):
        fields = {
            "event_id":row.event_id,"timestamp_ms":int(row.timestamp_ms),"visitor_id":row.visitor_id,
            "customer_id":row.customer_id,"event_type":row.event_type,"product_id":row.product_id,
            "quantity":int(row.quantity),"amount":float(row.amount),"discount":float(row.discount),
            "channel":row.channel,"latency_ms":int(row.latency_ms),"search_term":row.search_term,"store_id":row.store_id,
        }
        pipe.xadd(STREAM_KEY,fields,maxlen=20000,approximate=True)
    stream_ids.extend(pipe.execute())

print("Eventos en Stream:",r.xlen(STREAM_KEY))
print("Primer ID:",stream_ids[0],"Último ID:",stream_ids[-1])

In [ ]:
purchases = events.loc[events["event_type"]=="purchase"].copy()
total_revenue = round(float(purchases["amount"].sum()),2)
total_orders = len(purchases)
aov = round(total_revenue/total_orders,2)

r.mset({"lab:kpi:revenue:period":total_revenue,"lab:kpi:orders:period":total_orders,"lab:kpi:aov:period":aov})
r.set("lab:cache:dashboard:summary",json.dumps({"revenue":total_revenue,"orders":total_orders,"aov":aov}),ex=300)

print("Revenue:",r.get("lab:kpi:revenue:period"))
print("Orders:",r.get("lab:kpi:orders:period"))
print("TTL cache:",r.ttl("lab:cache:dashboard:summary"))

In [ ]:
bitmap_map = customers.set_index("customer_id")["bitmap_id"].to_dict()
for event_date,group in events.groupby("event_date"):
    active = sorted(set(group.loc[group["customer_id"]!="","customer_id"]))
    unique = sorted(set(group["visitor_id"]))
    pipe = r.pipeline(transaction=False)
    for customer_id in active:
        pipe.setbit(f"lab:active:{event_date}",int(bitmap_map[customer_id]),1)
    pipe.pfadd(f"lab:visitors:{event_date}",*unique)
    pipe.execute()

latest_date = max(events["event_date"])
print("Activos:",r.bitcount(f"lab:active:{latest_date}"))
print("Visitantes estimados:",r.pfcount(f"lab:visitors:{latest_date}"))

In [ ]:
revenue_by_product = purchases.groupby("product_id")["amount"].sum().sort_values(ascending=False)
r.zadd("lab:ranking:products:revenue",revenue_by_product.to_dict())

view_counts = events.loc[events["event_type"]=="view"].groupby("product_id").size().sort_values(ascending=False)
r.execute_command("CMS.INITBYPROB","lab:frequency:product_views",0.001,0.01)
args = ["CMS.INCRBY","lab:frequency:product_views"]
for product_id,count in view_counts.items():
    args.extend([product_id,int(count)])
r.execute_command(*args)

r.execute_command("TOPK.RESERVE","lab:trending:products",5,2000,7,0.925)
args = ["TOPK.INCRBY","lab:trending:products"]
for product_id,count in view_counts.items():
    args.extend([product_id,int(count)])
r.execute_command(*args)

print("Ranking:",r.zrevrange("lab:ranking:products:revenue",0,4,withscores=True))
print("Top-K:",r.execute_command("TOPK.LIST","lab:trending:products","WITHCOUNT"))

In [ ]:
EVENT_BLOOM_KEY = "lab:dedupe:events"
LATENCY_TDIGEST_KEY = "lab:latency:all"

r.execute_command("BF.RESERVE",EVENT_BLOOM_KEY,0.001,len(events)*2)
for start in range(0,len(events),500):
    r.execute_command("BF.MADD",EVENT_BLOOM_KEY,*events.iloc[start:start+500]["event_id"].tolist())

r.execute_command("CF.RESERVE","lab:blocked:transactions",10000)
r.execute_command("CF.ADD","lab:blocked:transactions","TX-FRAUD-0001")
r.execute_command("CF.ADD","lab:blocked:transactions","TX-FRAUD-0002")

r.execute_command("TDIGEST.CREATE",LATENCY_TDIGEST_KEY,"COMPRESSION",200)
latencies = events["latency_ms"].astype(int).tolist()
for start in range(0,len(latencies),500):
    r.execute_command("TDIGEST.ADD",LATENCY_TDIGEST_KEY,*latencies[start:start+500])

print("Evento conocido:",r.execute_command("BF.EXISTS",EVENT_BLOOM_KEY,events.iloc[0]["event_id"]))
print("Evento nuevo:",r.execute_command("BF.EXISTS",EVENT_BLOOM_KEY,"EVT-NEW"))
print("Percentiles:",r.execute_command("TDIGEST.QUANTILE",LATENCY_TDIGEST_KEY,0.50,0.95,0.99))

In [ ]:
RETENTION_MS = 90*24*60*60*1000
minute_revenue = purchases.groupby(["channel","minute"],as_index=False)["amount"].sum()
minute_revenue["timestamp_ms"] = minute_revenue["minute"].astype("int64") // 1_000_000

for channel in sorted(events["channel"].unique()):
    r.execute_command("TS.CREATE",f"lab:ts:revenue:{channel}","RETENTION",RETENTION_MS,"DUPLICATE_POLICY","SUM","LABELS","metric","revenue","channel",channel)

for channel,group in minute_revenue.groupby("channel"):
    pipe = r.pipeline(transaction=False)
    for row in group.itertuples(index=False):
        pipe.execute_command("TS.ADD",f"lab:ts:revenue:{channel}",int(row.timestamp_ms),float(row.amount))
    pipe.execute()

assert r.xlen(STREAM_KEY)==len(events)
assert int(r.get("lab:kpi:orders:period"))==len(purchases)
print("Carga validada")

# 10. Análisis con Redis y pandas

Redis responde consultas operativas. pandas presenta resultados pequeños. No descargues millones de claves para resolver una consulta que Redis puede ejecutar directamente.

In [ ]:
records = []
for key in r.scan_iter(match="lab:customer:*",count=100):
    item = r.hgetall(key)
    records.append({
        "customer_id":item["customer_id"],"name":item["name"],"city":item["city"],"segment":item["segment"],
        "purchases":int(item["purchases"]),"total_spent":float(item["total_spent"]),"churn_score":float(item["churn_score"]),
    })
customers_redis = pd.DataFrame(records).sort_values("customer_id")
display(customers_redis.head())
display(customers_redis.groupby("segment")["total_spent"].agg(["count","mean","median","sum"]).sort_values("sum",ascending=False))

In [ ]:
campaign_members = customers.sample(15,random_state=42)["customer_id"].tolist()
r.sadd("lab:campaign:retention",*campaign_members)
premium_campaign = sorted(r.sinter("lab:segment:premium","lab:campaign:retention"))
cohort = customers_redis.loc[customers_redis["customer_id"].isin(premium_campaign)].sort_values("total_spent",ascending=False)
print("Miembros:",r.scard("lab:campaign:retention"),"Premium:",len(premium_campaign))
display(cohort)

In [ ]:
ranking = pd.DataFrame(r.zrevrange("lab:ranking:products:revenue",0,-1,withscores=True),columns=["product_id","revenue"])
ranking = ranking.merge(products[["product_id","name","category","price"]],on="product_id",how="left")
ranking.insert(0,"rank",np.arange(1,len(ranking)+1))
display(ranking.head(10))

ax = ranking.head(10).set_index("name")["revenue"].sort_values().plot(kind="barh",figsize=(10,5),title="Productos con mayor ingreso")
ax.set_xlabel("Ingresos")
ax.set_ylabel("Producto")
plt.tight_layout()
plt.show()

In [ ]:
dates = sorted(events["event_date"].unique())
rows = []
for date in dates:
    rows.append({"date":date,"active_registered":r.bitcount(f"lab:active:{date}"),"unique_visitors_estimate":r.pfcount(f"lab:visitors:{date}")})
activity = pd.DataFrame(rows)
retention = [np.nan]
for previous,current in zip(dates[:-1],dates[1:]):
    r.bitop("AND","lab:tmp:retention",f"lab:active:{previous}",f"lab:active:{current}")
    base = r.bitcount(f"lab:active:{previous}")
    retention.append(r.bitcount("lab:tmp:retention")/base if base else np.nan)
r.delete("lab:tmp:retention")
activity["retention"] = retention
display(activity)

ax = activity.set_index("date")[["active_registered","unique_visitors_estimate"]].plot(marker="o",figsize=(11,5),title="Actividad diaria")
ax.set_ylabel("Usuarios")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
r.execute_command("PFMERGE","lab:visitors:period",*[f"lab:visitors:{date}" for date in dates])
estimated = r.pfcount("lab:visitors:period")
exact = events["visitor_id"].nunique()
print("Estimado:",estimated,"Exacto:",exact,"Error relativo:",abs(estimated-exact)/exact)

In [ ]:
result = r.execute_command("GEOSEARCH","lab:stores","FROMLONLAT",-3.6900,40.4300,"BYRADIUS",15,"km","WITHDIST","ASC")
rows = []
for member,distance in result:
    store_id = member.removeprefix("store:")
    metadata = r.hgetall(f"lab:store:{store_id}")
    rows.append({"store_id":store_id,"name":metadata["name"],"city":metadata["city"],"distance_km":float(distance)})
display(pd.DataFrame(rows))

In [ ]:
def pairs_to_dict(values):
    return dict(zip(values[0::2],values[1::2]))

def parse_search(response):
    total = int(response[0])
    rows = []
    for i in range(1,len(response),2):
        rows.append({"key":response[i],**pairs_to_dict(response[i+1])})
    return total,pd.DataFrame(rows)

response = r.execute_command(
    "FT.SEARCH",INDEX_NAME,"@category:{informatica} @price:[300 1500] @stock:[1 +inf]",
    "RETURN",5,"product_id","name","category","price","stock","SORTBY","price","ASC","DIALECT",2,
)
total,search_df = parse_search(response)
print("Coincidencias:",total)
display(search_df)

In [ ]:
text_response = r.execute_command("FT.SEARCH",INDEX_NAME,"ergonómica","RETURN",4,"product_id","name","category","price","DIALECT",2)
text_total,text_df = parse_search(text_response)
print("Coincidencias:",text_total)
display(text_df)

aggregate = r.execute_command(
    "FT.AGGREGATE",INDEX_NAME,"*","GROUPBY",1,"@category","REDUCE","COUNT",0,"AS","products",
    "REDUCE","AVG",1,"@price","AS","average_price","SORTBY",2,"@average_price","DESC","DIALECT",2,
)
aggregate_df = pd.DataFrame([pairs_to_dict(row) for row in aggregate[1:]])
display(aggregate_df)

In [ ]:
start_ms = int(START.value//1_000_000)
end_ms = int((END-pd.Timedelta(milliseconds=1)).value//1_000_000)
DAY_MS = 86_400_000
rows = []
for channel in sorted(events["channel"].unique()):
    points = r.execute_command("TS.RANGE",f"lab:ts:revenue:{channel}",start_ms,end_ms,"AGGREGATION","SUM",DAY_MS)
    for timestamp_ms,value in points:
        rows.append({"timestamp":pd.to_datetime(timestamp_ms,unit="ms",utc=True).tz_convert("Europe/Madrid"),"channel":channel,"revenue":float(value)})
revenue_ts = pd.DataFrame(rows).pivot_table(index="timestamp",columns="channel",values="revenue",aggfunc="sum",fill_value=0).sort_index()
display(revenue_ts)

ax = revenue_ts.plot(marker="o",figsize=(11,5),title="Ingresos diarios por canal")
ax.set_ylabel("Ingresos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
query_ids = products["product_id"].tolist()
cms_values = r.execute_command("CMS.QUERY","lab:frequency:product_views",*query_ids)
cms_df = pd.DataFrame({"product_id":query_ids,"cms_estimate":[int(x) for x in cms_values]}).merge(view_counts.rename("exact_views").reset_index(),on="product_id",how="left").fillna({"exact_views":0})
cms_df["exact_views"] = cms_df["exact_views"].astype(int)
cms_df["overestimation"] = cms_df["cms_estimate"]-cms_df["exact_views"]
display(cms_df.sort_values("exact_views",ascending=False))

quantiles = r.execute_command("TDIGEST.QUANTILE",LATENCY_TDIGEST_KEY,0.50,0.95,0.99)
print("t-digest:",quantiles)
print("pandas:",[events["latency_ms"].quantile(q) for q in [0.50,0.95,0.99]])

# 11. Pipelines, atomicidad y Lua

Un pipeline reduce viajes de red. `transaction=False` no crea una transacción. `transaction=True` utiliza `MULTI` y `EXEC`, pero Redis no ofrece rollback relacional.

In [ ]:
N = 500
delete_by_pattern(r,"lab:benchmark:*")

start = time.perf_counter()
for i in range(N):
    r.hset(f"lab:benchmark:direct:{i}",mapping={"value":i,"square":i*i})
direct = time.perf_counter()-start

start = time.perf_counter()
pipe = r.pipeline(transaction=False)
for i in range(N):
    pipe.hset(f"lab:benchmark:pipeline:{i}",mapping={"value":i,"square":i*i})
pipe.execute()
pipelined = time.perf_counter()-start

benchmark = pd.DataFrame([{"method":"direct","seconds":direct},{"method":"pipeline","seconds":pipelined}])
benchmark["operations_per_second"] = N/benchmark["seconds"]
display(benchmark)
print("Aceleración observada:",direct/pipelined)
delete_by_pattern(r,"lab:benchmark:*")

## 11.1. Compra atómica

La compra debe comprobar y descontar stock sin permitir que dos clientes consuman la última unidad. Separar lectura y escritura crea una condición de carrera.

El script:

1. lee stock;
2. valida producto y cantidad;
3. rechaza stock insuficiente;
4. descuenta unidades;
5. registra la operación en un Stream;
6. devuelve estado y stock restante.

In [ ]:
PURCHASE_SCRIPT = """
local stock = tonumber(redis.call('HGET', KEYS[1], ARGV[1]))
local quantity = tonumber(ARGV[2])
if not stock then return {-1, 0} end
if quantity <= 0 then return {-2, stock} end
if stock < quantity then return {0, stock} end
local remaining = redis.call('HINCRBY', KEYS[1], ARGV[1], -quantity)
redis.call('XADD', KEYS[2], '*', 'product_id', ARGV[1], 'quantity', ARGV[2], 'customer_id', ARGV[3], 'remaining_stock', remaining)
return {1, remaining}
"""

purchase = r.register_script(PURCHASE_SCRIPT)
status,remaining = map(int,purchase(keys=["lab:inventory:stock","lab:events:purchases:atomic"],args=["P001",2,"C1001"]))
labels = {1:"compra confirmada",0:"stock insuficiente",-1:"producto inexistente",-2:"cantidad inválida"}
print(labels[status],"Stock restante:",remaining)
print(r.xrevrange("lab:events:purchases:atomic",count=1))

## 11.2. Límite de solicitudes atómico

El contador establece TTL únicamente cuando crea la ventana.

In [ ]:
RATE_LIMIT_SCRIPT = """
local current = redis.call('INCR', KEYS[1])
if current == 1 then redis.call('EXPIRE', KEYS[1], ARGV[1]) end
return {current, redis.call('TTL', KEYS[1])}
"""

rate_limit = r.register_script(RATE_LIMIT_SCRIPT)
key = "lab:rate:user:C1001"
r.delete(key)
for request in range(1,6):
    current,ttl = rate_limit(keys=[key],args=[60])
    print(f"Solicitud {request}: contador={int(current)}, ttl={int(ttl)}")

# 12. Procesamiento con Streams

Un grupo distribuye entradas entre consumidores. Una entrada permanece pendiente hasta recibir `XACK`.

In [ ]:
GROUP = "analytics"
try:
    r.xgroup_create(STREAM_KEY,GROUP,id="0",mkstream=True)
except ResponseError as exc:
    if "BUSYGROUP" not in str(exc):
        raise

messages = r.xreadgroup(groupname=GROUP,consumername="analyst-1",streams={STREAM_KEY:">"},count=10,block=1000)
rows = []
for stream_name,entries in messages:
    for message_id,fields in entries:
        rows.append({"message_id":message_id,"event_id":fields["event_id"],"event_type":fields["event_type"],"product_id":fields["product_id"],"amount":float(fields["amount"])})
        r.xack(stream_name,GROUP,message_id)
display(pd.DataFrame(rows))
print("Pendientes:",r.execute_command("XPENDING",STREAM_KEY,GROUP))
print("Grupos:",r.execute_command("XINFO","GROUPS",STREAM_KEY))

In [ ]:
unacked = r.xreadgroup(groupname=GROUP,consumername="analyst-2",streams={STREAM_KEY:">"},count=1,block=1000)
print("Sin confirmar:",unacked)
print("Pendientes:",r.execute_command("XPENDING",STREAM_KEY,GROUP))
if unacked:
    stream_name,entries = unacked[0]
    message_id,_ = entries[0]
    r.xack(stream_name,GROUP,message_id)
print("Pendientes después de XACK:",r.execute_command("XPENDING",STREAM_KEY,GROUP))

# 13. Caso integrado

El panel combina String, Sorted Set, Bitmap, HyperLogLog, t-digest y Stream. No recorre el historial para calcular cada indicador.

In [ ]:
def operational_dashboard(client):
    latest = max(events["event_date"])
    q = client.execute_command("TDIGEST.QUANTILE",LATENCY_TDIGEST_KEY,0.50,0.95,0.99)
    return {
        "revenue":float(client.get("lab:kpi:revenue:period")),
        "orders":int(client.get("lab:kpi:orders:period")),
        "average_order_value":float(client.get("lab:kpi:aov:period")),
        "latest_date":latest,
        "active_registered":client.bitcount(f"lab:active:{latest}"),
        "unique_visitors":client.pfcount(f"lab:visitors:{latest}"),
        "top_products":client.zrevrange("lab:ranking:products:revenue",0,4,withscores=True),
        "latency_p50":float(q[0]),"latency_p95":float(q[1]),"latency_p99":float(q[2]),
        "stream_length":client.xlen(STREAM_KEY),
    }

dashboard = operational_dashboard(r)
print(json.dumps(dashboard,ensure_ascii=False,indent=2))

In [ ]:
commercial = ranking[["product_id","name","category","revenue"]].merge(cms_df[["product_id","exact_views","cms_estimate"]],on="product_id",how="outer").fillna(0)
commercial["revenue_per_view"] = np.where(commercial["exact_views"]>0,commercial["revenue"]/commercial["exact_views"],0)
display(commercial.sort_values(["exact_views","revenue"],ascending=[False,False]))

latency = events.groupby("channel")["latency_ms"].agg(requests="count",mean="mean",median="median",p95=lambda x:x.quantile(0.95),p99=lambda x:x.quantile(0.99),maximum="max").sort_values("p95",ascending=False)
display(latency)
ax = latency[["median","p95","p99"]].plot(kind="bar",figsize=(9,5),title="Latencia por canal")
ax.set_ylabel("Milisegundos")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 14. Práctica final

## Actividad 1. Cohorte comercial

Obtén los clientes que pertenecen al segmento `premium` y a la campaña `retention`. Recupera perfiles y ordénalos por gasto.

## Actividad 2. Alcance

Combina los HyperLogLog de los últimos siete días y compara la estimación con el valor exacto del DataFrame.

## Actividad 3. Catálogo

Busca productos de categoría `informatica` o `accesorios`, precio entre 100 y 1000 y stock positivo. Ordena por precio descendente.

## Actividad 4. Tiendas

Localiza las tres tiendas más cercanas a `(-3.7000, 40.4400)` dentro de veinte kilómetros.

## Actividad 5. Rendimiento

Obtén p50, p95 y p99 con t-digest. Compara con pandas e informa diferencias.

## Actividad 6. Streams

Crea el grupo `fraud-analysis`, lee cinco compras, confirma cuatro, inspecciona pendientes y confirma la última.

## Actividad 7. Modelado

Selecciona estructura y justifica:

1. número exacto de cupones usados por cliente;
2. usuarios únicos mensuales aproximados;
3. pedidos que requieren confirmación y reintento;
4. cinco búsquedas más frecuentes;
5. inventario con descuento atómico;
6. informes anuales con joins complejos.

## Solución de referencia: actividades 1 a 5

Ejecuta después de completar tu propuesta.

In [ ]:
# 1
ids = sorted(r.sinter("lab:segment:premium","lab:campaign:retention"))
rows = []
for customer_id in ids:
    p = r.hgetall(f"lab:customer:{customer_id}")
    rows.append({"customer_id":customer_id,"name":p["name"],"city":p["city"],"total_spent":float(p["total_spent"]),"churn_score":float(p["churn_score"])})
display(pd.DataFrame(rows).sort_values("total_spent",ascending=False))

# 2
last7 = dates[-7:]
r.execute_command("PFMERGE","lab:visitors:last7",*[f"lab:visitors:{d}" for d in last7])
est = r.pfcount("lab:visitors:last7")
exa = events.loc[events["event_date"].isin(last7),"visitor_id"].nunique()
print("HLL:",est,"Exacto:",exa,"Error:",abs(est-exa))

# 3
resp = r.execute_command("FT.SEARCH",INDEX_NAME,"(@category:{informatica|accesorios}) @price:[100 1000] @stock:[1 +inf]","RETURN",5,"product_id","name","category","price","stock","SORTBY","price","DESC","DIALECT",2)
_,df = parse_search(resp)
display(df)

# 4
near = r.execute_command("GEOSEARCH","lab:stores","FROMLONLAT",-3.7000,40.4400,"BYRADIUS",20,"km","WITHDIST","ASC","COUNT",3)
display(pd.DataFrame([{"store":m,"distance_km":float(d)} for m,d in near]))

# 5
probs = [0.50,0.95,0.99]
td = [float(x) for x in r.execute_command("TDIGEST.QUANTILE",LATENCY_TDIGEST_KEY,*probs)]
exact_q = [float(events["latency_ms"].quantile(q)) for q in probs]
result = pd.DataFrame({"quantile":probs,"tdigest":td,"pandas_exact":exact_q})
result["absolute_difference"] = (result["tdigest"]-result["pandas_exact"]).abs()
display(result)

## Solución de referencia: actividad 6

In [ ]:
FRAUD_GROUP = "fraud-analysis"
try:
    r.xgroup_create(STREAM_KEY,FRAUD_GROUP,id="0",mkstream=True)
except ResponseError as exc:
    if "BUSYGROUP" not in str(exc):
        raise

purchases_read = []
while len(purchases_read)<5:
    response = r.xreadgroup(groupname=FRAUD_GROUP,consumername="fraud-worker-1",streams={STREAM_KEY:">"},count=50,block=1000)
    if not response:
        break
    for stream_name,entries in response:
        for message_id,fields in entries:
            if fields["event_type"]=="purchase" and len(purchases_read)<5:
                purchases_read.append((stream_name,message_id,fields))
            else:
                r.xack(stream_name,FRAUD_GROUP,message_id)

for stream_name,message_id,_ in purchases_read[:4]:
    r.xack(stream_name,FRAUD_GROUP,message_id)
print("Pendientes tras cuatro:",r.execute_command("XPENDING",STREAM_KEY,FRAUD_GROUP))
if len(purchases_read)==5:
    stream_name,message_id,_ = purchases_read[4]
    r.xack(stream_name,FRAUD_GROUP,message_id)
print("Pendientes finales:",r.execute_command("XPENDING",STREAM_KEY,FRAUD_GROUP))

## Solución de referencia: actividad 7

| Necesidad | Estructura | Justificación |
|---|---|---|
| Cupones usados | Hash o String | Contador exacto |
| Usuarios únicos mensuales | HyperLogLog | Cardinalidad aproximada con memoria estable |
| Pedidos con reintento | Stream | Pendientes, consumidores y confirmación |
| Cinco búsquedas frecuentes | Top-K | Principales elementos con memoria acotada |
| Inventario atómico | Hash con Lua | Validación y descuento indivisibles |
| Informes anuales con joins | Data warehouse | Redis no es el destino adecuado para ese patrón histórico |

# 15. Vector Sets desde Python

Esta sección requiere Redis 8 en `6380`. El código detecta si está disponible.

In [ ]:
r8 = Redis(host=REDIS_HOST,port=REDIS8_PORT,decode_responses=True,socket_connect_timeout=2,socket_timeout=5)
try:
    VECTOR_READY = bool(r8.ping())
except ConnectionError:
    VECTOR_READY = False
print("Redis 8 disponible:",VECTOR_READY)

In [ ]:
vectors = {
    "P001":[0.95,0.85,0.10,0.05],"P002":[0.88,0.82,0.12,0.08],"P003":[0.08,0.40,0.95,0.45],
    "P004":[0.05,0.42,0.90,0.50],"P005":[0.30,0.70,0.12,0.18],"P006":[0.72,0.78,0.18,0.10],
}
if VECTOR_READY:
    delete_by_pattern(r8,"lab:product:vectors*")
    metadata = products.set_index("product_id").to_dict("index")
    for product_id,vector in vectors.items():
        r8.execute_command("VADD","lab:product:vectors","VALUES",len(vector),*vector,product_id)
        r8.execute_command("VSETATTR","lab:product:vectors",product_id,json.dumps({"category":metadata[product_id]["category"],"price":metadata[product_id]["price"]}))
    print(r8.execute_command("VSIM","lab:product:vectors","VALUES",4,0.90,0.80,0.15,0.10,"COUNT",4,"WITHSCORES"))
    print(r8.execute_command("VSIM","lab:product:vectors","VALUES",4,0.90,0.80,0.15,0.10,"COUNT",4,"FILTER",".price < 500","WITHSCORES"))
else:
    print("Inicia redis8-vector-lab para ejecutar esta sección")

# 16. Criterios de modelado

Antes de elegir una estructura, responde:

1. ¿Cuál es la clave natural de acceso?
2. ¿La consulta debe ser exacta?
3. ¿Necesitas recuperar miembros o únicamente contar?
4. ¿El orden importa?
5. ¿El dato caduca?
6. ¿La estructura crecerá sin límite?
7. ¿La operación requiere atomicidad?
8. ¿El dato representa estado actual, eventos o histórico?
9. ¿La aplicación conoce la clave o necesita buscar por campos?
10. ¿Qué ocurre si una actualización se pierde o se retrasa?

### Señales de diseño incorrecto

* JSON para contadores simples.
* Una clave por observación temporal sin retención.
* Lists para eventos críticos.
* `KEYS *` en producción.
* Streams sin estrategia de recorte.
* Sorted Sets sin limpieza.
* Bloom o Top-K para decisiones exactas.
* Descarga completa a pandas para una consulta pequeña.
* Duplicación sin definir la vista autoritativa.
* Lectura y escritura separadas cuando existe una condición de carrera.

Redis funciona bien como capa de estado operativo, caché, índice en tiempo real, contador, ranking, log de eventos y almacenamiento temporal de métricas. No sustituye un data warehouse, un lago de datos, un sistema histórico de auditoría ni un motor relacional para joins transaccionales complejos.

In [ ]:
checks = {
    "customer_hash":r.type("lab:customer:C1001")=="hash",
    "premium_set":r.type("lab:segment:premium")=="set",
    "revenue_ranking":r.type("lab:ranking:products:revenue")=="zset",
    "event_stream":r.type(STREAM_KEY)=="stream",
    "indexed_products":r.execute_command("FT.SEARCH",INDEX_NAME,"*","LIMIT",0,0,"DIALECT",2)[0]==len(products),
    "bloom_loaded":r.execute_command("BF.EXISTS",EVENT_BLOOM_KEY,events.iloc[0]["event_id"])==1,
    "timeseries_web":bool(r.execute_command("TS.INFO","lab:ts:revenue:web")),
}
checks_df = pd.DataFrame([{"check":name,"passed":passed} for name,passed in checks.items()])
display(checks_df)
if not checks_df["passed"].all():
    raise AssertionError("Fallos: "+", ".join(checks_df.loc[~checks_df["passed"],"check"]))
print("Laboratorio completado correctamente")

# 17. Referencias técnicas

* [Redis Stack en Docker](https://redis.io/docs/latest/operate/oss_and_stack/install/archive/install-stack/docker/)
* [Tipos de datos](https://redis.io/docs/latest/develop/data-types/)
* [Redis Streams](https://redis.io/docs/latest/develop/data-types/streams/)
* [Redis JSON](https://redis.io/docs/latest/develop/data-types/json/)
* [Redis Search](https://redis.io/docs/latest/develop/ai/search-and-query/)
* [Redis Time Series](https://redis.io/docs/latest/develop/data-types/timeseries/)
* [Estructuras probabilísticas](https://redis.io/docs/latest/develop/data-types/probabilistic/)
* [Vector Sets](https://redis.io/docs/latest/develop/data-types/vector-sets/)
* [redis-py](https://redis.io/docs/latest/develop/clients/redis-py/)